In [1]:

import pandas as pd
import plotly.express as px

df = px.data.gapminder()

print(f"Loaded: {len(df)} rows")
print(df.head())


Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [8]:

# Task 1 - Scatter Chart

import plotly.graph_objects as go

asia = df[
    (df["continent"] == "Asia") &
    (df["year"].isin([2002, 2007]))
].copy()

fig1 = px.scatter(
    asia,
    x="gdpPercap",
    y="lifeExp",
    color=asia["year"].astype(str),
    hover_name="country",
    log_x=True,
    color_discrete_map={
        "2002": "#F59E0B",
        "2007": "#2563EB"
    },
    title="Life Expectancy Increased Across Asia (2002–2007)",
    labels={
        "gdpPercap": "GDP per Capita (USD, log scale)",
        "lifeExp": "Life Expectancy (Years)",
        "color": "Year"
    }
)

# Connect each country's two points
for country in asia["country"].unique():
    points = asia[
        asia["country"] == country
    ].sort_values("year")

    fig1.add_trace(
        go.Scatter(
            x=points["gdpPercap"],
            y=points["lifeExp"],
            mode="lines",
            line=dict(color="lightgrey", width=1),
            showlegend=False,
            hoverinfo="skip"
        )
    )

fig1.update_layout(
    template="plotly_white",
    width=1000,
    height=600,
    legend_title_text="Year"
)

fig1.show()

import plotly.io as pio
pio.renderers.default = "iframe"
fig1.show()


In [12]:

# TASK 1 - Scatter Chart
# Life Expectancy Changes in Asia (2002-2007)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe"

# Load Gapminder dataset
df = px.data.gapminder()

# Filter Asia for 2002 and 2007
asia = df[
    (df["continent"] == "Asia") &
    (df["year"].isin([2002, 2007]))
].copy()

# Calculate average change in life expectancy
comparison = asia.pivot(
    index="country",
    columns="year",
    values="lifeExp"
)

changes = comparison[2007] - comparison[2002]
avg_change = changes.mean()
all_improved = (changes > 0).all()

# Create scatter chart
fig1 = go.Figure()

# Connect each country's 2002 and 2007 points
for country in asia["country"].unique():
    country_data = asia[
        asia["country"] == country
    ].sort_values("year")

    fig1.add_trace(
        go.Scatter(
            x=country_data["gdpPercap"],
            y=country_data["lifeExp"],
            mode="lines",
            line=dict(
                color="#BDBDBD",
                width=1.2
            ),
            showlegend=False,
            hoverinfo="skip"
        )
    )

# Add grey points for 2002 and blue points for 2007
for year, color in [
    (2002, "#D3D3D3"),
    (2007, "#3188C2")
]:
    year_data = asia[asia["year"] == year]

    fig1.add_trace(
        go.Scatter(
            x=year_data["gdpPercap"],
            y=year_data["lifeExp"],
            mode="markers",
            name=str(year),
            text=year_data["country"],
            marker=dict(
                color=color,
                size=11
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "GDP per Capita: $%{x:,.0f}<br>"
                "Life Expectancy: %{y:.1f} years"
                "<extra></extra>"
            )
        )
    )

# Highlight China, Iraq and Kuwait
for country in ["China", "Iraq", "Kuwait"]:
    point = asia[
        (asia["country"] == country) &
        (asia["year"] == 2007)
    ].iloc[0]

    fig1.add_annotation(
        x=point["gdpPercap"],
        y=point["lifeExp"],
        text=country,
        showarrow=True,
        arrowhead=0,
        arrowcolor="#3188C2",
        font=dict(
            color="#1676BA",
            size=14
        ),
        ax=30,
        ay=-30
    )

# Create insight title from actual data
if all_improved:
    title = (
        "Between 2002 and 2007 every Asian country "
        f"gained life expectancy — {avg_change:.1f} years on average"
    )
else:
    title = (
        "Life expectancy changed across Asia "
        f"by {avg_change:+.1f} years on average (2002–2007)"
    )

# Chart layout
fig1.update_layout(
    title=dict(
        text=title,
        font=dict(size=21)
    ),
    xaxis=dict(
        title="GDP per Capita (USD, log scale)",
        type="log",
        showgrid=True
    ),
    yaxis=dict(
        title="Life Expectancy (Years)",
        range=[40, 85],
        showgrid=True
    ),
    template="plotly_white",
    width=1400,
    height=650,
    legend=dict(
        orientation="h",
        x=0,
        y=1.08,
        title=None
    ),
    margin=dict(
        l=80,
        r=60,
        t=110,
        b=70
    )
)

fig1.show()


In [9]:

# Task 2 - Bubble Chart: Asia vs Rest of World

data_2007 = df[df["year"] == 2007].copy()

data_2007["Group"] = data_2007["continent"].apply(
    lambda x: "Asia" if x == "Asia" else "Other Continents"
)

fig2 = px.scatter(
    data_2007,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="Group",
    hover_name="country",
    log_x=True,
    size_max=55,
    color_discrete_map={
        "Asia": "#F59E0B",
        "Other Continents": "#BFC3C7"
    },
    category_orders={
        "Group": ["Other Continents", "Asia"]
    },
    title="Asia in 2007: Higher GDP Did Not Always Mean Longer Lives",
    labels={
        "gdpPercap": "GDP per Capita (USD, log scale)",
        "lifeExp": "Life Expectancy (Years)",
        "pop": "Population",
        "Group": "Region"
    },
    opacity=0.75
)

# Annotation highlighting India
fig2.add_annotation(
    x=2452,
    y=64.698,
    text="India: Large population,<br>lower life expectancy",
    showarrow=True,
    arrowhead=2,
    ax=110,
    ay=-65,
    bgcolor="white",
    bordercolor="#F59E0B",
    borderwidth=1
)

fig2.update_layout(
    template="plotly_white",
    width=1100,
    height=650
)

import plotly.io as pio
pio.renderers.default = "iframe"

fig2.show()
